# Milestone 7: Kernel fusion

## The idea

In the 5-kernel pipeline, every stage writes a full-size image to global memory, and the next stage immediately reads it back. Those in-between images exist only to pass data from one kernel to the next.

Minimum bytes each version must move through global memory **per pixel** (exact counts come in Milestone 9):

| Stage (5-kernel version E) | read | write |
|---|---|---|
| black level | 2 (uint16) | 4 (float) |
| demosaic | 4 | 12 (3 floats) |
| white balance | 12 | 12 |
| denoise | 12 | 12 |
| gamma | 12 | 3 (3 bytes) |
| **total** | | **≈ 85 bytes** |

**Fused version G:** read 2 bytes of raw input, write 3 bytes of output, so **≈ 5 bytes per pixel**. That's about **17× less memory traffic**. As a bonus, the in-between buffers aren't needed at all, so GPU memory use drops from about 740 MB to about 112 MB.

## How one kernel can do all five stages

Two stages need neighbours (demosaic and denoise), so a thread can't just compute its own pixel alone. The kernel uses the shared-memory tile idea from Milestone 6, twice:

1. **Phase 1:** the block loads a **36 × 12** Bayer tile (its 32 × 8 pixels plus a **2-pixel** halo) and applies **black level** while loading. `__syncthreads()`.
2. **Phase 2:** the block computes a **34 × 10** RGB tile (**1-pixel** halo) with **demosaic** and **white balance**. `__syncthreads()`.
3. **Phase 3:** each thread computes **denoise** and **gamma** for its own pixel and writes 3 bytes.

**Why a 2-pixel halo for the Bayer tile?** Denoise needs RGB values 1 pixel outside the block. Demosaicing *those* pixels needs Bayer values 1 pixel further out. 1 + 1 = 2.

**The price:** each block demosaics 34 × 10 = 340 pixels to produce 256 outputs, because the halo pixels are also computed by the neighbouring blocks. That's about 33% extra maths, traded for about 17× less memory traffic.

## Honest prediction

Fused should be clearly faster, but **not** 17×. The fused kernel still does all the maths (including 3 `powf` per pixel, plus the extra 33% of demosaicing), and its `__syncthreads()` phases mean threads sometimes wait. How close it gets tells us how much of the old pipeline's time was spent moving memory. **The measurement decides.**

**Runtime must be T4 GPU.**

## Step 1: Connect Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project

## Step 2: Write the new files

In [ ]:
%%writefile gpu_fused_kernel.h
// gpu_fused_kernel.h
// Milestone 7: all 5 ISP stages in ONE kernel.

#pragma once

#include <cstdint>

// Each block produces a 32 x 8 patch of output pixels (one per thread).
const int FUSED_TILE_WIDTH = 32;
const int FUSED_TILE_HEIGHT = 8;

// Denoise needs RGB values 1 pixel around the patch, and demosaicing THOSE
// needs Bayer values 1 pixel further out. So the two shared tiles are:
//   RGB tile:   halo 1  ->  34 x 10
//   Bayer tile: halo 2  ->  36 x 12
const int RGB_HALO = 1;
const int BAYER_HALO = 2;
const int RGB_TILE_WIDTH = FUSED_TILE_WIDTH + 2 * RGB_HALO;       // 34
const int RGB_TILE_HEIGHT = FUSED_TILE_HEIGHT + 2 * RGB_HALO;     // 10
const int BAYER_TILE_WIDTH = FUSED_TILE_WIDTH + 2 * BAYER_HALO;   // 36
const int BAYER_TILE_HEIGHT = FUSED_TILE_HEIGHT + 2 * BAYER_HALO; // 12

__global__ void fusedPipelineKernel(const uint16_t* bayerPixels, unsigned char* outputBytes,
                                    int imageWidth, int imageHeight,
                                    float blackLevel, float whiteLevel,
                                    float redGain, float greenGain, float blueGain);


In [ ]:
%%writefile gpu_fused_kernel.cu
// gpu_fused_kernel.cu
// All 5 stages in one kernel ("kernel fusion").
//
// Why: in the 5-kernel pipeline, every stage writes its full-size result to
// global memory and the next stage reads it straight back. Those in-between
// images (up to 270 MB each) exist only to pass data from one kernel to the
// next. Here, the in-between values live in shared memory and registers
// instead, and global memory sees only the raw input (read) and the final
// 8-bit output (written).
//
// The kernel works in three phases, separated by __syncthreads():
//   Phase 1: load Bayer tile (36 x 12), applying stage 1 (black level) on the way.
//   Phase 2: build RGB tile (34 x 10) with stage 2 (demosaic) and stage 3 (white balance).
//   Phase 3: each thread does stage 4 (denoise) and stage 5 (gamma) for its own
//            pixel and writes 3 bytes to global memory.
//
// The trade-off: the 1-pixel border of each RGB tile is also inside the
// neighbouring blocks' tiles, so those pixels are demosaiced twice (by both
// blocks). 34 x 10 = 340 demosaics for 256 output pixels: about 33% extra maths
// in exchange for far less memory traffic.

#include "gpu_fused_kernel.h"

static __device__ int reflectIndexFused(int index, int size) {
    if (index < 0) {
        return -index;
    }
    if (index >= size) {
        return 2 * size - 2 - index;
    }
    return index;
}

static __device__ int clampIndexFused(int index, int size) {
    if (index < 0) {
        return 0;
    }
    if (index >= size) {
        return size - 1;
    }
    return index;
}

// Reflect first (correct Bayer colour at the image edge, as on the CPU), then
// clamp. The clamp only matters for tile positions far outside the image in
// the last block of a row/column; their values are never used, but the index
// must still stay inside the buffer.
static __device__ int safeBayerIndex(int index, int size) {
    int reflectedIndex = reflectIndexFused(index, size);
    return clampIndexFused(reflectedIndex, size);
}

static __device__ unsigned char gammaToByteFused(float linearValue) {
    if (linearValue < 0.0f) {
        linearValue = 0.0f;
    }
    if (linearValue > 1.0f) {
        linearValue = 1.0f;
    }
    float gammaValue = powf(linearValue, 1.0f / 2.2f);
    float scaledValue = gammaValue * 255.0f + 0.5f;
    return static_cast<unsigned char>(scaledValue);
}

static __device__ float clampToOne(float value) {
    if (value > 1.0f) {
        return 1.0f;
    }
    return value;
}

__global__ void fusedPipelineKernel(const uint16_t* bayerPixels, unsigned char* outputBytes,
                                    int imageWidth, int imageHeight,
                                    float blackLevel, float whiteLevel,
                                    float redGain, float greenGain, float blueGain) {
    __shared__ float bayerTile[BAYER_TILE_HEIGHT][BAYER_TILE_WIDTH];
    __shared__ float rgbTile[3][RGB_TILE_HEIGHT][RGB_TILE_WIDTH];

    int blockStartColumn = blockIdx.x * FUSED_TILE_WIDTH;
    int blockStartRow = blockIdx.y * FUSED_TILE_HEIGHT;
    int threadNumber = threadIdx.y * FUSED_TILE_WIDTH + threadIdx.x;
    int threadsPerBlock = FUSED_TILE_WIDTH * FUSED_TILE_HEIGHT;

    // Image position of tile position (0, 0) for each tile.
    int bayerTileOriginRow = blockStartRow - BAYER_HALO;
    int bayerTileOriginColumn = blockStartColumn - BAYER_HALO;
    int rgbTileOriginRow = blockStartRow - RGB_HALO;
    int rgbTileOriginColumn = blockStartColumn - RGB_HALO;

    // ---- Phase 1: load the Bayer tile, applying black level correction ----
    // Same "thread t loads t, t+256, t+512 ..." pattern as Milestone 6.
    float usefulRange = whiteLevel - blackLevel;
    int bayerTileSize = BAYER_TILE_WIDTH * BAYER_TILE_HEIGHT;
    for (int loadNumber = threadNumber; loadNumber < bayerTileSize; loadNumber = loadNumber + threadsPerBlock) {
        int tileRow = loadNumber / BAYER_TILE_WIDTH;
        int tileColumn = loadNumber % BAYER_TILE_WIDTH;
        int imageRow = safeBayerIndex(bayerTileOriginRow + tileRow, imageHeight);
        int imageColumn = safeBayerIndex(bayerTileOriginColumn + tileColumn, imageWidth);

        float rawValue = static_cast<float>(bayerPixels[imageRow * imageWidth + imageColumn]);
        float normalisedValue = (rawValue - blackLevel) / usefulRange;
        if (normalisedValue < 0.0f) {
            normalisedValue = 0.0f;
        }
        if (normalisedValue > 1.0f) {
            normalisedValue = 1.0f;
        }
        bayerTile[tileRow][tileColumn] = normalisedValue;
    }
    __syncthreads();

    // ---- Phase 2: demosaic + white balance into the RGB tile ----
    int rgbTileSize = RGB_TILE_WIDTH * RGB_TILE_HEIGHT;
    for (int workNumber = threadNumber; workNumber < rgbTileSize; workNumber = workNumber + threadsPerBlock) {
        int tileRow = workNumber / RGB_TILE_WIDTH;
        int tileColumn = workNumber % RGB_TILE_WIDTH;

        // The denoise stage clamps at the image edge, so an RGB position
        // outside the image must hold the demosaiced value of the nearest
        // edge pixel. We demosaic THAT pixel.
        int imageRow = clampIndexFused(rgbTileOriginRow + tileRow, imageHeight);
        int imageColumn = clampIndexFused(rgbTileOriginColumn + tileColumn, imageWidth);

        // Where that pixel sits in the Bayer tile. Phase 1 stored reflected
        // edge values in the halo, so neighbours at +-1 are always available.
        int centreRow = imageRow - bayerTileOriginRow;
        int centreColumn = imageColumn - bayerTileOriginColumn;

        float centre = bayerTile[centreRow][centreColumn];
        float up = bayerTile[centreRow - 1][centreColumn];
        float down = bayerTile[centreRow + 1][centreColumn];
        float left = bayerTile[centreRow][centreColumn - 1];
        float right = bayerTile[centreRow][centreColumn + 1];
        float upLeft = bayerTile[centreRow - 1][centreColumn - 1];
        float upRight = bayerTile[centreRow - 1][centreColumn + 1];
        float downLeft = bayerTile[centreRow + 1][centreColumn - 1];
        float downRight = bayerTile[centreRow + 1][centreColumn + 1];

        float crossAverage = (up + down + left + right) / 4.0f;
        float diagonalAverage = (upLeft + upRight + downLeft + downRight) / 4.0f;
        float horizontalAverage = (left + right) / 2.0f;
        float verticalAverage = (up + down) / 2.0f;

        // Colour is decided by the IMAGE position (RGGB), not the tile position.
        bool isEvenRow = (imageRow % 2 == 0);
        bool isEvenColumn = (imageColumn % 2 == 0);

        float redValue = 0.0f;
        float greenValue = 0.0f;
        float blueValue = 0.0f;

        if (isEvenRow && isEvenColumn) {
            redValue = centre;
            greenValue = crossAverage;
            blueValue = diagonalAverage;
        } else if (isEvenRow && !isEvenColumn) {
            redValue = horizontalAverage;
            greenValue = centre;
            blueValue = verticalAverage;
        } else if (!isEvenRow && isEvenColumn) {
            redValue = verticalAverage;
            greenValue = centre;
            blueValue = horizontalAverage;
        } else {
            redValue = diagonalAverage;
            greenValue = crossAverage;
            blueValue = centre;
        }

        // Stage 3, white balance, applied straight away.
        rgbTile[0][tileRow][tileColumn] = clampToOne(redValue * redGain);
        rgbTile[1][tileRow][tileColumn] = clampToOne(greenValue * greenGain);
        rgbTile[2][tileRow][tileColumn] = clampToOne(blueValue * blueGain);
    }
    __syncthreads();

    // ---- Phase 3: denoise + gamma for this thread's own pixel ----
    int column = blockStartColumn + threadIdx.x;
    int row = blockStartRow + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    const float gaussianWeights[3][3] = {
        {1.0f, 2.0f, 1.0f},
        {2.0f, 4.0f, 2.0f},
        {1.0f, 2.0f, 1.0f}
    };
    const float weightSum = 16.0f;

    int centreTileRow = threadIdx.y + RGB_HALO;
    int centreTileColumn = threadIdx.x + RGB_HALO;
    int pixelIndex = row * imageWidth + column;

    for (int channel = 0; channel < 3; channel++) {
        float weightedTotal = 0.0f;
        for (int rowOffset = -1; rowOffset <= 1; rowOffset++) {
            for (int columnOffset = -1; columnOffset <= 1; columnOffset++) {
                float neighbourValue = rgbTile[channel][centreTileRow + rowOffset][centreTileColumn + columnOffset];
                float weight = gaussianWeights[rowOffset + 1][columnOffset + 1];
                weightedTotal = weightedTotal + neighbourValue * weight;
            }
        }
        float denoisedValue = weightedTotal / weightSum;
        outputBytes[pixelIndex * 3 + channel] = gammaToByteFused(denoisedValue);
    }
}


In [ ]:
%%writefile gpu_fusion_experiment_main.cu
// gpu_fusion_experiment_main.cu
// Milestone 7: compares the best 5-kernel pipeline (E: planar, 32x8 blocks)
// with the single fused kernel (G). Both are timed from "first kernel starts"
// to "last kernel ends", 1 warm-up + 10 averaged runs, measured E G E G.

#include <iomanip>
#include <iostream>
#include <vector>

#include "cuda_helpers.h"
#include "gpu_buffers.h"
#include "gpu_fused_kernel.h"
#include "gpu_kernels.h"
#include "gpu_planar_kernels.h"
#include "isp_io.h"

const int TIMED_RUN_COUNT = 10;

// Launches the five separate kernels (configuration E from Milestone 5).
static void launchFiveKernels(DeviceBuffers& buffers, const ImageMetadata& metadata,
                              dim3 gridShape, dim3 blockShape) {
    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;

    blackLevelKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceNormalised,
                                                imageWidth, imageHeight,
                                                metadata.blackLevel, metadata.whiteLevel);
    checkCudaError(cudaGetLastError(), "launching blackLevelKernel");
    demosaicPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                                    imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching demosaicPlanarKernel");
    whiteBalancePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                        metadata.whiteBalanceRed,
                                                        metadata.whiteBalanceGreen,
                                                        metadata.whiteBalanceBlue);
    checkCudaError(cudaGetLastError(), "launching whiteBalancePlanarKernel");
    denoisePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                                   imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching denoisePlanarKernel");
    gammaPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceDenoised, buffers.deviceOutput,
                                                 imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching gammaPlanarKernel");
}

// Launches the single fused kernel. It reads the raw Bayer data and writes the
// final bytes; the in-between buffers are not used at all.
static void launchFusedKernel(DeviceBuffers& buffers, const ImageMetadata& metadata,
                              dim3 gridShape, dim3 blockShape) {
    fusedPipelineKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceOutput,
                                                   metadata.imageWidth, metadata.imageHeight,
                                                   metadata.blackLevel, metadata.whiteLevel,
                                                   metadata.whiteBalanceRed,
                                                   metadata.whiteBalanceGreen,
                                                   metadata.whiteBalanceBlue);
    checkCudaError(cudaGetLastError(), "launching fusedPipelineKernel");
}

// Times one version: 1 warm-up run (ignored), then the average of 10 runs.
static float averageMilliseconds(bool useFusedKernel, DeviceBuffers& buffers,
                                 const ImageMetadata& metadata,
                                 dim3 gridShape, dim3 blockShape,
                                 cudaEvent_t startEvent, cudaEvent_t stopEvent) {
    float totalMilliseconds = 0.0f;

    for (int runNumber = 0; runNumber <= TIMED_RUN_COUNT; runNumber++) {
        checkCudaError(cudaEventRecord(startEvent), "recording start event");
        if (useFusedKernel) {
            launchFusedKernel(buffers, metadata, gridShape, blockShape);
        } else {
            launchFiveKernels(buffers, metadata, gridShape, blockShape);
        }
        checkCudaError(cudaEventRecord(stopEvent), "recording stop event");
        checkCudaError(cudaEventSynchronize(stopEvent), "waiting for the run");

        float runMilliseconds = 0.0f;
        checkCudaError(cudaEventElapsedTime(&runMilliseconds, startEvent, stopEvent),
                       "reading run time");

        // Run 0 is the warm-up: measured, but not added to the average.
        if (runNumber > 0) {
            totalMilliseconds = totalMilliseconds + runMilliseconds;
        }
    }
    return totalMilliseconds / TIMED_RUN_COUNT;
}

int main() {
    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int pixelCount = metadata.imageWidth * metadata.imageHeight;
    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }

    DeviceBuffers buffers;
    allocateDeviceBuffers(buffers, pixelCount);
    checkCudaError(cudaMemcpy(buffers.deviceBayer, bayerPixels.data(), buffers.bayerBytes,
                              cudaMemcpyHostToDevice), "copying Bayer data to the GPU");

    cudaEvent_t startEvent;
    cudaEvent_t stopEvent;
    checkCudaError(cudaEventCreate(&startEvent), "creating start event");
    checkCudaError(cudaEventCreate(&stopEvent), "creating stop event");

    // Both versions use 32 x 8 blocks, so they can share one launch shape.
    dim3 blockShape(FUSED_TILE_WIDTH, FUSED_TILE_HEIGHT);
    int blocksAcross = (metadata.imageWidth + FUSED_TILE_WIDTH - 1) / FUSED_TILE_WIDTH;
    int blocksDown = (metadata.imageHeight + FUSED_TILE_HEIGHT - 1) / FUSED_TILE_HEIGHT;
    dim3 gridShape(blocksAcross, blocksDown);

    std::cout << std::fixed << std::setprecision(3);
    std::cout << "Average time for all ISP stages, " << TIMED_RUN_COUNT << " warm runs each:" << std::endl;

    float fiveKernelFirst = averageMilliseconds(false, buffers, metadata, gridShape, blockShape, startEvent, stopEvent);
    std::cout << "  E  5 separate kernels : " << fiveKernelFirst << " ms" << std::endl;
    float fusedFirst = averageMilliseconds(true, buffers, metadata, gridShape, blockShape, startEvent, stopEvent);
    std::cout << "  G  1 fused kernel     : " << fusedFirst << " ms" << std::endl;
    float fiveKernelSecond = averageMilliseconds(false, buffers, metadata, gridShape, blockShape, startEvent, stopEvent);
    std::cout << "  E  again              : " << fiveKernelSecond << " ms" << std::endl;
    float fusedSecond = averageMilliseconds(true, buffers, metadata, gridShape, blockShape, startEvent, stopEvent);
    std::cout << "  G  again              : " << fusedSecond << " ms" << std::endl;

    float fiveKernelAverage = (fiveKernelFirst + fiveKernelSecond) / 2.0f;
    float fusedAverage = (fusedFirst + fusedSecond) / 2.0f;
    std::cout << "Speed-up of fused over 5 kernels: " << fiveKernelAverage / fusedAverage << "x" << std::endl;

    // GPU memory each version needs. The fused kernel only needs input and output.
    double fiveKernelMegabytes = (buffers.bayerBytes + buffers.normalisedBytes
                                  + 2 * buffers.rgbFloatBytes + buffers.outputBytes) / 1000000.0;
    double fusedMegabytes = (buffers.bayerBytes + buffers.outputBytes) / 1000000.0;
    std::cout << "GPU memory needed: 5 kernels " << fiveKernelMegabytes
              << " MB, fused " << fusedMegabytes << " MB" << std::endl;

    // The last run was the fused kernel, so deviceOutput holds its result.
    std::vector<unsigned char> outputBytes(buffers.outputBytes);
    checkCudaError(cudaMemcpy(outputBytes.data(), buffers.deviceOutput, buffers.outputBytes,
                              cudaMemcpyDeviceToHost), "copying output to the CPU");

    checkCudaError(cudaEventDestroy(startEvent), "destroying start event");
    checkCudaError(cudaEventDestroy(stopEvent), "destroying stop event");
    freeDeviceBuffers(buffers);

    bool writeOk = writeRgbFile("output/gpu_fused_output.bin", outputBytes);
    if (!writeOk) {
        return 1;
    }
    std::cout << "Saved output/gpu_fused_output.bin" << std::endl;
    return 0;
}


## Step 3: Compile
**Expected:** no output.

In [ ]:
!nvcc -O2 -arch=sm_75 -o /content/gpu_fusion gpu_fusion_experiment_main.cu gpu_fused_kernel.cu gpu_planar_kernels.cu gpu_buffers.cu gpu_pointwise_kernels.cu gpu_neighbourhood_kernels.cu cuda_helpers.cu isp_io.cpp
!g++ -O2 -std=c++17 -o /content/compare_outputs compare_outputs.cpp isp_io.cpp

## Step 4: Run (CHECKPOINT 1)
**Expected:** four timing lines (E, G, E again, G again), a speed-up line, a GPU memory line (about 740 MB vs 112 MB), and `Saved output/gpu_fused_output.bin`.

**Paste the whole output to me.**

In [ ]:
!/content/gpu_fusion

## Step 5: Check against the CPU (CHECKPOINT 2)
**Expected:** `RESULT: PASS`. Since the fused kernel does the same maths in the same order, the result should match the earlier GPU versions.

In [ ]:
!/content/compare_outputs output/cpu_output.bin output/gpu_fused_output.bin

## Step 6: Look at the final photo (optional)
The output of your fastest pipeline.

In [ ]:
!python view_output.py output/gpu_fused_output.bin output/gpu_fused_output.png
from PIL import Image
from IPython.display import display
previewImage = Image.open('output/gpu_fused_output.png')
previewImage.thumbnail((1000, 1000))
display(previewImage)